# 08 X与Y关联分析
分别在Y1、Y2各自冻结测试侧取X交集：预期1,051／1,050行。两者测试样本不同，不能强行共用一组。Y为全部连续预测分数，不只保留弱标签正例。

首次使用：在python目录的终端执行 python -m pip install -r requirements.txt，然后执行 jupyter lab。

In [1]:
from pathlib import Path
import sys
# 从python目录或项目根启动均可；没有固定机器路径
PYTHON_DIR = Path.cwd() if (Path.cwd() / 'analysis_helpers.py').exists() else Path.cwd() / 'python'
if not (PYTHON_DIR / 'analysis_helpers.py').exists():
    raise FileNotFoundError('请从lasso论文或lasso论文/python目录打开此Notebook')
sys.path.insert(0, str(PYTHON_DIR.resolve()))
try:
    import analysis_helpers as h
except ImportError as exc:
    raise ImportError("缺少依赖，请在python目录运行：python -m pip install -r requirements.txt；安装后重启内核") from exc
print(h.runtime_check())

{'python': '3.12.14', 'numpy': '2.3.5', 'scipy': '1.17.0', 'scikit-learn': '1.8.0'}


## 1 从规则重新计算X，从原文重新预测Y
所有来源都是旧版冻结数据，不读取new目录。

In [2]:
x_rows, environment = h.compute_x()
scores = {name: h.score_target(name) for name in ['Y1','Y2']}
print({name: len(table) for name,table in scores.items()})

{'Y1': 9133, 'Y2': 9133}


## 2 运行实际聚类检验
OLS仅含截距和X/0.1；按ENV聚类的双侧WCR11检验，99,999次Rademacher抽样，seed=20261013。95%区间为CR2＋Satterthwaite，自助法p和区间不是同一种检验的互相反演。这里真实重跑，不读取旧p。

In [3]:
results = {}
analysis_rows = {}
for name in ['Y1','Y2']:
    analysis_rows[name], results[name] = h.analyze_target(name, scores[name], x_rows)
    print(name, '完成:', results[name]['n_used'], '行，', results[name]['n_clusters'], '个ENV')

Y1 完成: 1051 行， 26 个ENV
Y2 完成: 1050 行， 26 个ENV


## 3 查看结果
β单位是X每增加0.1时，预测分数的百分点变化。原始p未作多重检验校正；这两个目标来自历史探索，不能将其当作事先只检验两次。正向点估计不等于显著关联，更不是因果效应。

In [4]:
import pandas as pd
summary = pd.DataFrame([{
    '目标': name, 'N': r['n_used'], 'ENV': r['n_clusters'],
    'β_预测分数百分点每X加0.1': r['effect_score_percentage_points_per_0_1_x'],
    'WCR11原始p': r['p_raw'],
    'CR2_95%下界': r['cr2_ci95_score_percentage_points'][0],
    'CR2_95%上界': r['cr2_ci95_score_percentage_points'][1],
    'CR2自由度': r['cr2_satterthwaite_df']
} for name,r in results.items()])
display(summary)

,目标,N,ENV,β_预测分数百分点每X加0.1,WCR11原始p,CR2_95%下界,CR2_95%上界,CR2自由度
0,Y1,1051,26,0.183979,0.21257,-0.152254,0.520212,6.012730
1,Y2,1050,26,0.150461,0.18781,-0.108109,0.409030,6.050988


## 4 可选保存本次实际结果
默认只显示；开启后写入各目标关联分析数据目录。

In [5]:
WRITE_OUTPUT = False
if WRITE_OUTPUT:
    import json
    for name,r in results.items():
        folder = h.DATA / h.TARGETS[name][2]
        (folder / '本次关联结果.json').write_text(json.dumps(r,ensure_ascii=False,indent=2),encoding='utf-8')
        analysis_rows[name].to_csv(folder / '本次关联输入.csv',index=False,encoding='utf-8-sig')